# 01_diagnose_combo_boxes

원본 Colab 셀에서 분리. (`#@title [진단] 조합코드로 전체(원본) 이미지 찾아서 박스 전부 표시`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [매 세션 3] dataset_74 zip 복원
import os                                                     # 경로 확인 도구

ZIP = os.path.join(PROJ_ROOT, "dataset_74_5fold.zip")         # 74클래스 5-fold zip(드라이브)
print("zip 존재:", os.path.exists(ZIP))                        # True 확인

!cp "$ZIP" /content/                                           # 드라이브 → 로컬 복사(속도 위해)
!unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74  # 압축 해제(-o=덮어쓰기)
print("복원 fold:", sorted(d for d in os.listdir("/content/dataset_74") if d.startswith("fold")))

In [ ]:
#@title [진단] 조합코드로 전체(원본) 이미지 찾아서 박스 전부 표시
import os, json                                            # 경로·JSON 로드 도구
import matplotlib.pyplot as plt                            # 이미지 시각화 도구
import matplotlib.patches as patches                       # 박스 그리기 도구
from PIL import Image                                       # 이미지 로드 도구

TARGET_COMBO = "K-011220-027777-028763"                    # 확인할 조합코드 (끝 -03 같은 크롭번호는 뺌)
FOLD = 2                                                    # 크롭 제목에 있던 fold 번호
SPLIT = "train"                                             # train/valid 중 어디서 나온 크롭인지 (모르면 둘 다 검색)
DATASET_DIR = "/content/dataset_74"                         # 74클래스 데이터셋 루트

def find_and_show(combo, fold, split):
    img_dir = f"{DATASET_DIR}/fold{fold}/{split}"           # 이미지+annotation 있는 폴더
    ann_path = f"{img_dir}/_annotations.coco.json"          # COCO annotation 경로
    with open(ann_path) as f:                                # annotation 파일 열기
        coco = json.load(f)                                  # json → dict

    cat_names = {c["id"]: c["name"] for c in coco["categories"]}  # category_id → 이름 매핑

    match_imgs = [im for im in coco["images"]                # 조합코드로 시작하는 파일명 전부 찾기
                  if im["file_name"].startswith(combo)]

    if not match_imgs:                                       # 없으면 반대 split도 시도하라고 안내
        print(f"{split}에 없음. SPLIT을 바꿔서 다시 시도하세요.")
        return

    for im in match_imgs:                                    # 같은 조합에 여러 각도 이미지 있을 수 있음
        fn = im["file_name"]                                 # 파일명
        img = Image.open(os.path.join(img_dir, fn)).convert("RGB")  # 전체(원본) 이미지 로드

        anns = [a for a in coco["annotations"] if a["image_id"] == im["id"]]  # 이 이미지의 박스 전부

        fig, ax = plt.subplots(figsize=(6,8))                # 캔버스 준비
        ax.imshow(img)                                       # 전체 이미지 표시
        for a in anns:                                       # 박스 전부 순회
            x, y, w, h = a["bbox"]                            # 박스 좌표
            ax.add_patch(patches.Rectangle((x,y), w, h,      # 빨간 사각형으로 표시
                         linewidth=1.5, edgecolor="red", facecolor="none"))
            ax.text(x, y-5, cat_names[a["category_id"]],      # 박스 위에 클래스명 표시
                     color="red", fontsize=7)
        ax.set_title(fn, fontsize=8)                          # 제목에 전체 파일명 표시
        ax.axis("off")
        plt.tight_layout()
        plt.show()

find_and_show(TARGET_COMBO, FOLD, SPLIT)                     # 실행